In [8]:
import tensorflow as tf
import numpy as np
import urllib.request

# ------------------------------------------------
# 1. Download a large text corpus from a guaranteed stable public URL
# ------------------------------------------------

# Using a highly reliable, permanent raw link to Alice in Wonderland from NLTK's official repository
url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/gutenberg/carroll-alice.zip"
zip_path = "alice.zip"

# Download the zip file
req = urllib.request.Request(
    "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/gutenberg.zip",
    headers={'User-Agent': 'Mozilla/5.0'}
)

print("Downloading corpus...")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/gutenberg.zip",
    zip_path
)

# Extract Carroll's Alice in Wonderland
import zipfile
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("gutenberg_extracted")

with open("gutenberg_extracted/gutenberg/carroll-alice.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text.lower()

# Use a manageable portion of the book
text = text[:100000]

print("Corpus length:", len(text))

# ------------------------------------------------
# 2. Create character vocabulary
# ------------------------------------------------

chars = sorted(set(text))

char_to_id = {c: i for i, c in enumerate(chars)}
id_to_char = {i: c for i, c in enumerate(chars)}

data = np.array([char_to_id[c] for c in text])

# ------------------------------------------------
# 3. Create input sequences and target characters
# ------------------------------------------------

seq_len = 60

X, y = [], []

for i in range(len(data) - seq_len):
    X.append(data[i:i + seq_len])
    y.append(data[i + seq_len])

X = np.array(X)
y = np.array(y)

# ------------------------------------------------
# 4. Build RNN model
# ------------------------------------------------

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(seq_len,)),
    tf.keras.layers.Embedding(len(chars), 64),
    tf.keras.layers.SimpleRNN(128),
    tf.keras.layers.Dense(len(chars), activation='softmax')
])

# ------------------------------------------------
# 5. Compile model
# ------------------------------------------------

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ------------------------------------------------
# 6. Train model
# ------------------------------------------------

model.fit(
    X, y,
    epochs=15,
    batch_size=128
)

# ------------------------------------------------
# 7. Generate new text
# ------------------------------------------------

seed = "alice "
input_seq = [char_to_id[c] for c in seed]

temperature = 0.8

for _ in range(300):

    pred = model.predict(
        np.array([input_seq[-seq_len:]]),
        verbose=0
    )[0]

    # Apply temperature for varied predictions
    pred = np.log(pred + 1e-8) / temperature
    probabilities = np.exp(pred)
    probabilities /= np.sum(probabilities)

    # Randomly select next character
    next_id = np.random.choice(
        len(chars), p=probabilities
    )

    next_char = id_to_char[next_id]

    seed += next_char
    input_seq.append(next_id)

# ------------------------------------------------
# 8. Display generated text
# ------------------------------------------------

print("\nGenerated Text:\n")
print(seed)

Corpus length: 100000
Epoch 1/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 50s 62ms/step - accuracy: 0.3237 - loss: 2.4284
Epoch 2/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 49s 63ms/step - accuracy: 0.4303 - loss: 1.9933
Epoch 3/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 50s 64ms/step - accuracy: 0.4692 - loss: 1.8397
Epoch 4/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 51s 65ms/step - accuracy: 0.4937 - loss: 1.7428
Epoch 5/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 49s 63ms/step - accuracy: 0.5103 - loss: 1.6710
Epoch 6/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 83s 64ms/step - accuracy: 0.5238 - loss: 1.6138
Epoch 7/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 81s 63ms/step - accuracy: 0.5364 - loss: 1.5687
Epoch 8/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 50s 64ms/step - accuracy: 0.5454 - loss: 1.5325
Epoch 9/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 50s 64ms/step - accuracy: 0.5537 - loss: 1.5010
Epoch 10/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 49s 62ms/step - accuracy: 0.5600 - loss: 1.4762
Epoch 11/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 50s 64ms/step - accuracy: 0.5663 - loss: 1.4525
Epoch 12